# Project Setup: 3D Deep Learning with PyTorch with LUNA Dataset

## Preparing a project

## Data loading

Task to solve in this project setup setp:
- Loading and processing raw data files
- Implementing a Python class to represent our data
- Converting our data into a format usable by PyTorch
- Visualizing the training and validation data

### Data Pre-processing

The first target is to split the data in a training and validation set. LUNA16 provides us with annotations and candidate data.

In [4]:
import csv
annotations = []
candidates = []
with open('/media/lennart/LaCie/datasets/LUNA16/candidates.csv') as f:
    candidates = f.readlines()

with open('/media/lennart/LaCie/datasets/LUNA16/annotations.csv') as f:
    annotations = f.readlines()

print(f"lenght candidtes: {len(candidates)}, length annotations: {len(annotations)}")

lenght candidtes: 551066, length annotations: 1187


Meaning 551,066 coordinates can be considered as being a nodule, while 1,187 are actually nodules.

#### Unifying annotation and candidate data
Stitching candidate and annotation data together.

In [6]:
DATASET_PATH = '/media/lennart/LaCie/datasets/LUNA16/'

In [7]:
from collections import namedtuple
CandidateInfoTuple = namedtuple(
'CandidateInfoTuple','isNodule_bool, diameter_mm, series_uid, center_xyz',
)

In [ ]:
@functools.lru_cache(1)
def getCandidateInfoList(requireOnDisk_bool=True):
    mhd_list = glob.glob(os.path.join(PATH_TO_DATASET, 'subset*/*.mhd'))
    presentOnDisk_set = {os.path.split(p)[-1][:4] for p in mhd_list} # all the data that is present on disk

    daimeter_dict = {}
    with open(os.path.join(PATH_TO_DATASET, 'annotations.csv'), "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]
            annotationCenter_xyz = tuple([float(x) for x in row[1:4]])
            annotationDiameter_mm = float(row[4])
            diameter_dict.setdefault(series_uid, []).append(
                (annotationCenter_xyz, annotationDiameter_mm)
            )

    with open(os.path.join(PATH_TO_DATASET, 'candidates.csv'), "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]

            if series_uid not in presentOnDisk_set and requireOnDisk_bool:
                continue

            isNodule_bool = bool(int(row[4]))
            candidateCenter_xyz = tuple([float(x) for x in row[1:4]])
            candidateDiameter_mm = 0.0
            
            for annotation_tup in diameter_dict.get(series_uid, []):
                annotationCenter_xyz, annotationDiameter_mm = annotation_tup
                
                for i in range(3):
                    delta_mm = abs(candidateCenter_xyz[i] - annotationCenter_xyz[i])
                    
                    if delta_mm > annotationDiameter_mm / 4: # candidate and center should not be too far apart
                        break
                    else:
                        candidateDiameter_mm = annotationDiameter_mm
                        break
                            
            candidateInfo_list.append(CandidateInfoTuple(
                isNodule_bool,
                candidateDiameter_mm,
                series_uid,
                candidateCenter_xyz,
            ))
            candidateInfo_list.sort(reverse=True) # samples starting with largest first
            return candidateInfo_list
                

#### Parsing the domain-specific data (CT-scans)

In [ ]:
import SimpleITK as sitk

class Ct:
    def __init__(self, series_uid):
        mhd_path = glob.glob(
            os.path.join(DATASET_PATH, 'subset*/{}.mhd'.format(series_uid))
        )[0]
        ct_mhd = sitk.ReadImage(mhd_path)
        ct_a = np.array(sitk.GetArrayFromImage(ct_mhd), dtype=np.float32)
        ct_a.clip(-1000, 1000, ct_a) # In LUNA air is -1,000 HU, water is 0 HU and bone is 1,000 HU
        self.series_uid = series_uid
        self.hu_a = ct_a
        
        self.origin_xyz = XyzTuple(*ct_mhd.GetOrigin())
        self.vxSize_xyz = XyzTuple(*ct_mhd.GetSpacing())
        self.direction_a = np.array(ct_mhd.GetDirection()).reshape(3, 3)

    def getRawCandidate(self, center_xyz, width_irc):
        center_irc = xyz2irc(
            center_xyz,
            self.origin_xyz,
            self.vxSize_xyz,
            self.direction_a,
        )
        slice_list = []
        for axis, center_val in enumerate(center_irc):
            start_ndx = int(round(center_val - width_irc[axis]/2))
            end_ndx = int(start_ndx + width_irc[axis])
            slice_list.append(slice(start_ndx, end_ndx))
        ct_chunk = self.hu_a[tuple(slice_list)]
        return ct_chunk, center_irc

#### Coordinate System Transformation

In [ ]:
IrcTuple = collections.namedtuple('IrcTuple', ['index', 'row', 'col'])
XyzTuple = collections.namedtuple('XyzTuple', ['x', 'y', 'z'])

def irc2xyz(coord_irc, origin_xyz, vxSize_xyz, direction_a):
    cri_a = np.array(coord_irc)[::-1] # Swapping the order
    origin_a = np.array(origin_xyz)
    vxSize_a = np.array(vxSize_xyz)
    coords_xyz = (direction_a @ (cri_a * vxSize_a)) + origin_a
    return XyzTuple(*coords_xyz)
    
def xyz2irc(coord_xyz, origin_xyz, vxSize_xyz, direction_a):
    origin_a = np.array(origin_xyz)
    vxSize_a = np.array(vxSize_xyz)
    coord_a = np.array(coord_xyz)
    cri_a = ((coord_a - origin_a) @ np.linalg.inv(direction_a)) / vxSize_a
    cri_a = np.round(cri_a)
    return IrcTuple(int(cri_a[2]), int(cri_a[1]), int(cri_a[0]))

#### PyTorch Dataset implementation

In [ ]:
class LunaDataset(Dataset):
    def __init__(self,
                 val_stride=0,
                 isValSet_bool=None,
                 series_uid=None,
                ):
        self.candidateInfo_list = copy.copy(getCandidateInfoList())
        if series_uid: # For debugging or visualization
            self.candidateInfo_list = [
                x for x in self.candidateInfo_list if x.series_uid == series_uid
            ]
        if isValSet_bool:
            assert val_stride > 0, val_stride
            self.candidateInfo_list = self.candidateInfo_list[::val_stride]
            assert self.candidateInfo_list
        elif val_stride > 0:
            del self.candidateInfo_list[::val_stride]
            assert self.candidateInfo_list
            
    def __len__(self):
        return len(self.candidateInfo_list)

    def __getitem__(self, ndx):
        candidateInfo_tup = self.candidateInfo_list[ndx]
        width_irc = (32, 48, 48)
        candidate_a, center_irc = getCtRawCandidate(
            candidateInfo_tup.series_uid,
            candidateInfo_tup.center_xyz,
            width_irc,
        )
        candidate_t = torch.from_numpy(candidate_a)
        candidate_t = candidate_t.to(torch.float32)
        candidate_t = candidate_t.unsqueeze(0)
        pos_t = torch.tensor([
            not candidateInfo_tup.isNodule_bool,
            candidateInfo_tup.isNodule_bool
        ], dtype=torch.long,)
        return (
           candidate_t, 1((CO10-1))
           pos_t, 1((CO10-2))
           candidateInfo_tup.series_uid,
           torch.tensor(center_irc),
    )
            
    

In [ ]:
@functools.lru_cache(1, typed=True)
def getCt(series_uid):
    return Ct(series_uid)
    
@raw_cache.memoize(typed=True)
def getCtRawCandidate(series_uid, center_xyz, width_irc):
    ct = getCt(series_uid)
    ct_chunk, center_irc = ct.getRawCandidate(center_xyz, width_irc)
    return ct_chunk, center_irc

### Segmentation

### Grouping

### Classification